## CrewAI vs AutoGen 개념 비교 및 코드 예시

01~03번 노트북에서는 LangGraph로 orchestrator/sub-agent, sequential/hierarchical, 메시지 전달 구조를 **직접 그래프 노드와 라우팅 코드로** 만들어봤다. CrewAI와 AutoGen은 이런 패턴들을 **미리 만들어진 추상화**로 제공하는 고수준 멀티 에이전트 프레임워크다. 같은 개념을 각 프레임워크가 어떻게 캡슐화하는지 비교해본다.

| | **CrewAI** | **AutoGen (AG2)** |
|---|---|---|
| 핵심 패러다임 | **역할(Role) 기반 협업**: Agent + Task + Crew | **대화(Conversation) 기반 협업**: 여러 ConversableAgent가 메시지를 주고받음 |
| 오케스트레이션 단위 | `Process.sequential` / `Process.hierarchical` (내장) | `GroupChatManager`가 매 라운드 다음 발언자를 선택 |
| 우리가 만든 것과 대응 | `Process.sequential` ≈ [02번] Sequential 그래프<br>`Process.hierarchical` ≈ [02번] manager 노드 | `GroupChatManager.select_speaker` ≈ [02번] manager 노드<br>Agent 간 메시지 ≈ [03번] Message 구조 |
| Human-in-the-loop | 지원 약함 (주로 자동 실행 전제) | `UserProxyAgent`로 사람이 대화에 직접 참여 가능 |
| 코드 실행 | Tool로 감싸서 사용 | `code_execution_config`로 에이전트가 직접 코드 실행 가능 |
| 정의 방식 | 선언적 (역할/목표/배경스토리를 채우는 방식) | 대화형 (system_message로 페르소나를 주고 메시지를 주고받음) |
| 적합한 경우 | 업무를 역할별로 명확히 나눌 수 있는 작업 (조사→작성→검토 파이프라인 등) | 자유로운 토론/브레인스토밍, 사람 개입이 필요한 워크플로 |

두 프레임워크 모두 우리가 LangGraph로 직접 구현했던 개념(순차 실행, 계층적 위임, 메시지 교환)을 그대로 담고 있다 — 다만 그래프의 노드·엣지를 직접 그리는 대신, **역할(CrewAI)** 또는 **대화 규칙(AutoGen)**이라는 더 높은 추상화 레벨에서 선언한다. 대신 라우팅 로직을 세밀하게 커스터마이즈하고 싶다면(예: 02번의 재작업 루프 조건) LangGraph처럼 낮은 수준의 제어권이 필요해진다.

> 이 노트북은 `crewai`, `pyautogen`(또는 `ag2`) 패키지가 필요하다. 두 패키지는 의존성 충돌이 잦으므로, 이전 [12_CrewAI 협업 에이전트 구현](../03_LangChain%20활용/12_CrewAI%20협업%20에이전트%20구현.ipynb) 노트북과 마찬가지로 **별도 가상환경 + 커널 재시작**을 권장한다.

In [1]:
# !pip install crewai
# !pip install autogen   

# 설치 후 Jupyter 커널을 재시작한다

In [2]:
import os
from dotenv import load_dotenv

# .env 파일에서 API 키 불러오기
load_dotenv("C:/env/.env")

True

### [1] CrewAI: 역할(Role) 기반 협업

CrewAI의 3요소는 다음과 같다.

- **Agent**: `role`(역할), `goal`(목표), `backstory`(페르소나)로 정의되는 전문가. LangGraph 노드 함수 하나에 해당한다.
- **Task**: Agent가 수행할 구체적 작업. `context`로 다른 Task의 결과물을 입력으로 넘길 수 있다 — [01번] orchestrator가 sub-task 사이에 결과를 전달하던 것과 같은 역할.
- **Crew**: Agent와 Task들을 묶고, `process`로 실행 순서를 결정한다.

In [3]:
from crewai import Agent, Crew, Process, Task

researcher = Agent(
    role="시장 조사원",
    goal="주어진 주제에 대한 핵심 트렌드 3가지를 조사한다",
    backstory="여러 산업 리포트를 분석해온 경험 많은 시장 조사 전문가다.",
    llm="gpt-4o-mini",
    verbose=True,
)

writer = Agent(
    role="보고서 작성자",
    goal="조사 결과를 경영진이 읽기 좋은 짧은 보고서로 정리한다",
    backstory="복잡한 조사 내용을 간결한 비즈니스 문서로 압축하는 데 능숙한 작가다.",
    llm="gpt-4o-mini",
    verbose=True,
)

research_task = Task(
    description="'{topic}'에 대한 최신 트렌드 3가지를 조사하고 각각을 한 문장으로 요약하라.",
    expected_output="트렌드 3개, 각각 한 문장 요약",
    agent=researcher,
)

write_task = Task(
    description="조사 결과를 바탕으로 경영진 보고용 3문단 요약 보고서를 작성하라.",
    expected_output="3문단으로 구성된 보고서",
    agent=writer,
    context=[research_task],  # research_task의 출력이 이 Task의 입력으로 자동 전달된다
)

# process=Process.sequential -> [02번] Sequential 그래프와 동일한 개념: 순서 고정, 되돌아가지 않음
sequential_crew = Crew(
    agents=[researcher, writer],
    tasks=[research_task, write_task],
    process=Process.sequential,
    verbose=True,
)

# Jupyter 커널은 이미 자체 이벤트 루프를 실행 중이므로 동기 kickoff()는
# "invoked synchronously from within a running event loop" 에러를 낸다.
# kickoff_async()를 await로 호출해 우회한다.
sequential_result = await sequential_crew.kickoff_async(inputs={"topic": "온디바이스 AI 반도체"})
print(sequential_result)

╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.15.14                                                                                       │
│  Latest version:  1.15.16                                                                                       │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: b6269aba-88d9-4d31-8179-a18d1be5a544                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: '온디바이스 AI 반도체'에 대한 최신 트렌드 3가지를 조사하고 각각을 한 문장으로 요약하라.                  │
│  ID: 01afa0e1-d9f6-4cce-bcb7-afe5957829d4                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: 시장 조사원                                                                                             │
│                                                                                                                 │
│  Task: '온디바이스 AI 반도체'에 대한 최신 트렌드 3가지를 조사하고 각각을 한 문장으로 요약하라.                  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: 보고서 작성자                                                                                           │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **경영진 보고서 요약**                                                                                         │
│                                                                                                                 │
│  최근 연구에 따르면, 온디바이스 AI 반도체는 사용자 개인정보 보호와 데이터 보안을 강화하는 데 중요한 역할을      │
│  하고 있습니다. 해당 기술은 데이터 전송을 최소화함으로써, 사용자의 민감한 정보가 외부로 유출될 위험을 줄이고    │
│  있습니다. 이는 기업의 신뢰도 향상과 고객 만족도 증가로 이어질 수 있는 중요한 요소입니다.                       │
│                                                                                                                 │
│  또한, 최신 온디바이스 AI 반도체는 저전력 소비를 통해 우수한 성능을 발휘하고 있어 다양한 휴대용 기기에서의      │
│  활용도가 높아지고 있습니다. 이로 인해, 사용자는 전력 소모를 걱정하지 않고도 높은 성능을 경험할 수 있으며,      │
│  이는 제품 경쟁력을 강화하는 데 기여합니다.                                                                     │
│                                                                                                                 │
│  마지막으로, 온디바이스 AI 기술의 모바일 기기 통합은 사용자 경험을 획기적으로 개선하고 있습니다. 실시간 데이터  │
│  처리 기능은 사용자에게 즉각적인 피드백을 제공하여 만족도를 높이며, 결과적으로 기업의 영업 성과에 긍정적인      │
│  영향을 미칠 수 있습니다. 이러한 기술적 발전이 지속적으로 이루어진다면, 시장에서의 경쟁 우위를 더욱 확고히 할   │
│  수 있을 것입니다.                                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 조사 결과를 바탕으로 경영진 보고용 3문단 요약 보고서를 작성하라.                                         │
│  Agent: 보고서 작성자                                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: b6269aba-88d9-4d31-8179-a18d1be5a544                                                                       │
│  Final Output: **경영진 보고서 요약**                                                                           │
│                                                                                                                 │
│  최근 연구에 따르면, 온디바이스 AI 반도체는 사용자 개인정보 보호와 데이터 보안을 강화하는 데 중요한 역할을      │
│  하고 있습니다. 해당 기술은 데이터 전송을 최소화함으로써, 사용자의 민감한 정보가 외부로 유출될 위험을 줄이고    │
│  있습니다. 이는 기업의 신뢰도 향상과 고객 만족도 증가로 이어질 수 있는 중요한 요소입니다.                       │
│                                                                                                                 │
│  또한, 최신 온디바이스 AI 반도체는 저전력 소비를 통해 우수한 성능을 발휘하고 있어 다양한 휴대용 기기에서의      │
│  활용도가 높아지고 있습니다. 이로 인해, 사용자는 전력 소모를 걱정하지 않고도 높은 성능을 경험할 수 있으며,      │
│  이는 제품 경쟁력을 강화하는 데 기여합니다.                                                                     │
│                                                                                                                 │
│  마지막으로, 온디바이스 AI 기술의 모바일 기기 통합은 사용자 경험을 획기적으로 개선하고 있습니다. 실시간 데이터  │
│  처리 기능은 사용자에게 즉각적인 피드백을 제공하여 만족도를 높이며, 결과적으로 기업의 영업 성과에 긍정적인      │
│  영향을 미칠 수 있습니다. 이러한 기술적 발전이 지속적으로 이루어진다면, 시장에서의 경쟁 우위를 더욱 확고히 할   │
│  수 있을 것입니다.                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

**경영진 보고서 요약**

최근 연구에 따르면, 온디바이스 AI 반도체는 사용자 개인정보 보호와 데이터 보안을 강화하는 데 중요한 역할을 하고 있습니다. 해당 기술은 데이터 전송을 최소화함으로써, 사용자의 민감한 정보가 외부로 유출될 위험을 줄이고 있습니다. 이는 기업의 신뢰도 향상과 고객 만족도 증가로 이어질 수 있는 중요한 요소입니다.

또한, 최신 온디바이스 AI 반도체는 저전력 소비를 통해 우수한 성능을 발휘하고 있어 다양한 휴대용 기기에서의 활용도가 높아지고 있습니다. 이로 인해, 사용자는 전력 소모를 걱정하지 않고도 높은 성능을 경험할 수 있으며, 이는 제품 경쟁력을 강화하는 데 기여합니다.

마지막으로, 온디바이스 AI 기술의 모바일 기기 통합은 사용자 경험을 획기적으로 개선하고 있습니다. 실시간 데이터 처리 기능은 사용자에게 즉각적인 피드백을 제공하여 만족도를 높이며, 결과적으로 기업의 영업 성과에 긍정적인 영향을 미칠 수 있습니다. 이러한 기술적 발전이 지속적으로 이루어진다면, 시장에서의 경쟁 우위를 더욱 확고히 할 수 있을 것입니다.


╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

`Process.hierarchical`을 쓰면 [02번] 노트북에서 직접 작성했던 `manager_node`를 CrewAI가 대신 만들어준다. `manager_llm`(또는 `manager_agent`)이 각 Task를 어떤 Agent에게 맡길지, 결과가 부족하면 누구에게 다시 맡길지를 자동으로 판단한다 — 우리가 손으로 짠 `Decision` 구조화 출력 + 라우팅 로직이 프레임워크 내부로 들어간 것이다.

In [4]:
reviewer = Agent(
    role="검토자",
    goal="보고서에 근거 없는 주장이나 어색한 표현이 없는지 검토한다",
    backstory="사실 확인과 교정에 엄격한 편집자다.",
    llm="gpt-4o-mini",
    verbose=True,
)

# process=Process.hierarchical -> manager가 런타임에 다음 담당자를 결정 (고정된 tasks 순서가 없어도 됨)
hierarchical_crew = Crew(
    agents=[researcher, writer, reviewer],
    tasks=[research_task, write_task],
    process=Process.hierarchical,
    manager_llm="gpt-4o-mini",  # 이 매니저가 [02번]의 manager_node 역할을 대신한다
    verbose=True,
)

# 위 셀과 동일한 이유로 kickoff_async()를 await로 호출한다.
hierarchical_result = await hierarchical_crew.kickoff_async(inputs={"topic": "온디바이스 AI 반도체"})
print(hierarchical_result)

╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.15.14                                                                                       │
│  Latest version:  1.15.16                                                                                       │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 3f57ab59-b593-4a51-b102-289587ea5431                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: '온디바이스 AI 반도체'에 대한 최신 트렌드 3가지를 조사하고 각각을 한 문장으로 요약하라.                  │
│  ID: 01afa0e1-d9f6-4cce-bcb7-afe5957829d4                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│  Task: '온디바이스 AI 반도체'에 대한 최신 트렌드 3가지를 조사하고 각각을 한 문장으로 요약하라.                  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: delegate_work_to_coworker                                                                                │
│  Args: {'task': "Research the latest trends in 'On-Device AI Semiconductors' and summarize each trend in one    │
│  sentence.", 'context': 'The task requires you to find the most up-to-date trends regarding on-devi...          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ### 경영진 보고서                                                                                              │
│                                                                                                                 │
│  #### 엣지 컴퓨팅에 대한 수요 증가                                                                              │
│  최근 엣지 컴퓨팅에 대한 수요가 급격히 증가하고 있습니다. 이는 IoT 기기의 성장이 가속화됨에 따라 발생하는       │
│  필수적인 변화로, 데이터 처리를 더욱 신속하게 수행할 수 있는 온디바이스 AI 반도체의 필요성이 부각되고           │
│  있습니다. 이러한 반도체는 데이터를 클라우드가 아닌 디바이스 자체에서 처리함으로써 지연 시간을 줄이고 효율성을  │
│  높여, 다양한 IoT 애플리케이션에서 중요한 역할을 하고 있습니다.                                                 │
│                                                                                                                 │
│  #### 에너지 효율성 향상                                                                                        │
│  반도체 기술의 발전은 에너지 효율성을 크게 향상시키고 있습니다. 새로운 반도체 솔루션은 배터리 수명을 늘리고,    │
│  에너지 소비를 최소화하는 데 기여합니다. 이는 전력 소모를 줄이는 동시에, IoT 기기를 통해 수집되는 데이터의      │
│  처리 능력을 극대화하여 지속 가능한 기술 환경을 조성하는 데 도움을 줍니다. 효율적인 에너지 사용은 기업의 운영   │
│  비용 절감뿐만 아니라, 사용자 경험 향상에도 긍정적인 영향을 미칩니다.                                           │
│                                                                                                                 │
│  #### 개인정보 보호 및 보안                                                                                     │
│  오늘날 개인정보 보호와 보안의 중요성이 더욱 강조되고 있는 가운데, 온디바이스 AI 반도체는 사용자 신뢰를 높이는  │
│  데 중요한 역할을 하고 있습니다. 이러한 반도체는 데이터를 디바이스 내에서 안전하게 처리하고 저장하기 때문에     │
│  외부 해킹 위험을 줄이며, 데이터 보호 규약에 대한 준수를 쉽게 합니다. 사용자에게는 더욱 안전한 환경을           │
│  제공하고, 기업 측면에서도 규제를 준수하는 데 필수적입니다. 이는 기업의 브랜드 신뢰도를 높이고 장기적인 고객    │
│  관계 구축에 기여합니다.                                                                                        │
│                                                                                                                 │
│  이 보고서는 엣지 컴퓨팅의 성장과 함께 온디바이스 AI 반도체의 필요성, 에너지 효율성 증진의 중요성, 그리고       │
│  개인정보 보호에 관한 최신 동향을 종합적으로 다루었습니다. 이와 같은 기술적 발전은 기업의 혁신과 경쟁력 강화에  │
│  매우 중요한 요소입니다.                                                                                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 조사 결과를 바탕으로 경영진 보고용 3문단 요약 보고서를 작성하라.                                         │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 3f57ab59-b593-4a51-b102-289587ea5431                                                                       │
│  Final Output: ### 경영진 보고서                                                                                │
│                                                                                                                 │
│  #### 엣지 컴퓨팅에 대한 수요 증가                                                                              │
│  최근 엣지 컴퓨팅에 대한 수요가 급격히 증가하고 있습니다. 이는 IoT 기기의 성장이 가속화됨에 따라 발생하는       │
│  필수적인 변화로, 데이터 처리를 더욱 신속하게 수행할 수 있는 온디바이스 AI 반도체의 필요성이 부각되고           │
│  있습니다. 이러한 반도체는 데이터를 클라우드가 아닌 디바이스 자체에서 처리함으로써 지연 시간을 줄이고 효율성을  │
│  높여, 다양한 IoT 애플리케이션에서 중요한 역할을 하고 있습니다.                                                 │
│                                                                                                                 │
│  #### 에너지 효율성 향상                                                                                        │
│  반도체 기술의 발전은 에너지 효율성을 크게 향상시키고 있습니다. 새로운 반도체 솔루션은 배터리 수명을 늘리고,    │
│  에너지 소비를 최소화하는 데 기여합니다. 이는 전력 소모를 줄이는 동시에, IoT 기기를 통해 수집되는 데이터의      │
│  처리 능력을 극대화하여 지속 가능한 기술 환경을 조성하는 데 도움을 줍니다. 효율적인 에너지 사용은 기업의 운영   │
│  비용 절감뿐만 아니라, 사용자 경험 향상에도 긍정적인 영향을 미칩니다.                                           │
│                                                                                                                 │
│  #### 개인정보 보호 및 보안                                                                                     │
│  오늘날 개인정보 보호와 보안의 중요성이 더욱 강조되고 있는 가운데, 온디바이스 AI 반도체는 사용자 신뢰를 높이는  │
│  데 중요한 역할을 하고 있습니다. 이러한 반도체는 데이터를 디바이스 내에서 안전하게 처리하고 저장하기 때문에     │
│  외부 해킹 위험을 줄이며, 데이터 보호 규약에 대한 준수를 쉽게 합니다. 사용자에게는 더욱 안전한 환경을           │
│  제공하고, 기업 측면에서도 규제를 준수하는 데 필수적입니다. 이는 기업의 브랜드 신뢰도를 높이고 장기적인 고객    │
│  관계 구축에 기여합니다.                                                                                        │
│                                                                                                                 │
│  이 보고서는 엣지 컴퓨팅의 성장과 함께 온디바이스 AI 반도체의 필요성, 에너지 효율성 증진의 중요성, 그리고       │
│  개인정보 보호에 관한 최신 동향을 종합적으로 다루었습니다. 이와 같은 기술적 발전은 기업의 혁신과 경쟁력 강화에  │
│  매우 중요한 요소입니다.                                                                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

### 경영진 보고서

#### 엣지 컴퓨팅에 대한 수요 증가
최근 엣지 컴퓨팅에 대한 수요가 급격히 증가하고 있습니다. 이는 IoT 기기의 성장이 가속화됨에 따라 발생하는 필수적인 변화로, 데이터 처리를 더욱 신속하게 수행할 수 있는 온디바이스 AI 반도체의 필요성이 부각되고 있습니다. 이러한 반도체는 데이터를 클라우드가 아닌 디바이스 자체에서 처리함으로써 지연 시간을 줄이고 효율성을 높여, 다양한 IoT 애플리케이션에서 중요한 역할을 하고 있습니다.

#### 에너지 효율성 향상
반도체 기술의 발전은 에너지 효율성을 크게 향상시키고 있습니다. 새로운 반도체 솔루션은 배터리 수명을 늘리고, 에너지 소비를 최소화하는 데 기여합니다. 이는 전력 소모를 줄이는 동시에, IoT 기기를 통해 수집되는 데이터의 처리 능력을 극대화하여 지속 가능한 기술 환경을 조성하는 데 도움을 줍니다. 효율적인 에너지 사용은 기업의 운영 비용 절감뿐만 아니라, 사용자 경험 향상에도 긍정적인 영향을 미칩니다.

#### 개인정보 보호 및 보안
오늘날 개인정보 보호와 보안의 중요성이 더욱 강조되고 있는 가운데, 온디바이스 AI 반도체는 사용자 신뢰를 높이는 데 중요한 역할을 하고 있습니다. 이러한 반도체는 데이터를 디바이스 내에서 안전하게 처리하고 저장하기 때문에 외부 해킹 위험을 줄이며, 데이터 보호 규약에 대한 준수를 쉽게 합니다. 사용자에게는 더욱 안전한 환경을 제공하고, 기업 측면에서도 규제를 준수하는 데 필수적입니다. 이는 기업의 브랜드 신뢰도를 높이고 장기적인 고객 관계 구축에 기여합니다. 

이 보고서는 엣지 컴퓨팅의 성장과 함께 온디바이스 AI 반도체의 필요성, 에너지 효율성 증진의 중요성, 그리고 개인정보 보호에 관한 최신 동향을 종합적으로 다루었습니다. 이와 같은 기술적 발전은 기업의 혁신과 경쟁력 강화에 매우 중요한 요소입니다.


╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

### [2] AutoGen (AG2): 대화(Conversation) 기반 협업

AutoGen은 Agent를 "역할이 채워진 존재"가 아니라 **메시지를 주고받는 대화 참여자(ConversableAgent)**로 다룬다.

- **AssistantAgent**: `system_message`로 페르소나가 주어진 LLM 기반 에이전트.
- **UserProxyAgent**: 사람 또는 자동화된 실행 주체 역할 — `human_input_mode`로 사람이 매 턴 개입할지, 아니면 자동 진행할지 정한다.
- 두 에이전트가 `initiate_chat`으로 대화를 시작하면, 서로에게 응답하며 대화가 이어진다 — [03번] 노트북의 `Message(sender, receiver, content)` 교환과 본질적으로 같은 구조다.

In [6]:
import autogen

llm_config = {
    "config_list": [{"model": "gpt-4o-mini", "api_key": os.environ["OPENAI_API_KEY"]}],
    "temperature": 0,
}

assistant = autogen.AssistantAgent(
    name="Researcher",
    system_message="너는 시장 조사 전문가다. 요청받은 주제의 핵심 트렌드를 간결하게 정리해서 답한다.",
    llm_config=llm_config,
)

user_proxy = autogen.UserProxyAgent(
    name="User",
    human_input_mode="NEVER",       # 사람 개입 없이 자동 진행
    code_execution_config=False,     # 코드 실행 기능은 사용하지 않음
    max_consecutive_auto_reply=1,
)

user_proxy.initiate_chat(assistant, message="온디바이스 AI 반도체 트렌드 3가지를 알려줘.")

User (to Researcher):

온디바이스 AI 반도체 트렌드 3가지를 알려줘.

--------------------------------------------------------------------------------
Researcher (to User):

온디바이스 AI 반도체 트렌드 3가지는 다음과 같습니다:

1. **에너지 효율성 향상**: 온디바이스 AI 반도체는 배터리 수명을 연장하고 전력 소비를 최소화하기 위해 에너지 효율성을 극대화하는 방향으로 발전하고 있습니다. 이를 통해 모바일 기기와 IoT 디바이스에서의 AI 처리 성능이 향상되고 있습니다.

2. **모델 경량화 및 최적화**: AI 모델을 경량화하고 최적화하여 온디바이스에서 실행할 수 있도록 하는 기술이 중요해지고 있습니다. 이는 하드웨어 자원이 제한된 환경에서도 고성능 AI 기능을 제공할 수 있게 합니다.

3. **프라이버시 및 데이터 보안 강화**: 온디바이스 AI는 데이터가 클라우드로 전송되지 않고 로컬에서 처리되기 때문에 사용자 프라이버시와 데이터 보안이 강화됩니다. 이는 특히 개인 정보 보호가 중요한 분야에서 큰 장점으로 작용하고 있습니다.

--------------------------------------------------------------------------------
User (to Researcher):



--------------------------------------------------------------------------------
Researcher (to User):

추가 질문이나 다른 주제에 대해 궁금한 점이 있으시면 언제든지 말씀해 주세요!

--------------------------------------------------------------------------------

>>>>>>>> TERMINATING RUN (3b3a5580-b566-4bf9-ae9e-455509dab15a): Maxim

ChatResult(chat_id=178757290609408717279720908667535520695, chat_history=[{'content': '온디바이스 AI 반도체 트렌드 3가지를 알려줘.', 'role': 'assistant', 'name': 'User'}, {'content': '온디바이스 AI 반도체 트렌드 3가지는 다음과 같습니다:\n\n1. **에너지 효율성 향상**: 온디바이스 AI 반도체는 배터리 수명을 연장하고 전력 소비를 최소화하기 위해 에너지 효율성을 극대화하는 방향으로 발전하고 있습니다. 이를 통해 모바일 기기와 IoT 디바이스에서의 AI 처리 성능이 향상되고 있습니다.\n\n2. **모델 경량화 및 최적화**: AI 모델을 경량화하고 최적화하여 온디바이스에서 실행할 수 있도록 하는 기술이 중요해지고 있습니다. 이는 하드웨어 자원이 제한된 환경에서도 고성능 AI 기능을 제공할 수 있게 합니다.\n\n3. **프라이버시 및 데이터 보안 강화**: 온디바이스 AI는 데이터가 클라우드로 전송되지 않고 로컬에서 처리되기 때문에 사용자 프라이버시와 데이터 보안이 강화됩니다. 이는 특히 개인 정보 보호가 중요한 분야에서 큰 장점으로 작용하고 있습니다.', 'role': 'user', 'name': 'Researcher'}, {'content': '', 'role': 'assistant', 'name': 'User'}, {'content': '추가 질문이나 다른 주제에 대해 궁금한 점이 있으시면 언제든지 말씀해 주세요!', 'role': 'user', 'name': 'Researcher'}], summary='추가 질문이나 다른 주제에 대해 궁금한 점이 있으시면 언제든지 말씀해 주세요!', cost={'usage_including_cached_inference': {'total_cost': 0.00020265, 'gpt-4o-mini-2024-07-18': {'cost': 0.00020265, 'prompt_tokens': 355, 'co

#### GroupChat: 3인 이상의 대화형 협업

`GroupChatManager`는 매 라운드 "다음 발언자가 누구여야 하는가"를 LLM으로 판단한다 — [02번] `manager_node`가 매번 `Decision`을 내려 다음 노드를 정하던 것과 정확히 같은 역할이다. 차이는 우리는 규칙을 프롬프트에 직접 적었지만, `GroupChatManager`는 `speaker_selection_method`(`"auto"`/`"round_robin"`/`"random"`/커스텀 함수)로 전략을 선택할 수 있다는 점이다.

In [7]:
writer = autogen.AssistantAgent(
    name="Writer",
    system_message="너는 보고서 작성자다. Researcher의 조사 내용을 3문단 보고서로 정리한다.",
    llm_config=llm_config,
)

critic = autogen.AssistantAgent(
    name="Critic",
    system_message=(
        "너는 검토자다. Writer의 보고서에 근거 없는 주장이 있으면 구체적으로 지적하라. "
        "문제가 없으면 'APPROVED'라고만 답하라."
    ),
    llm_config=llm_config,
)

admin = autogen.UserProxyAgent(
    name="Admin",
    human_input_mode="NEVER",
    code_execution_config=False,
)

groupchat = autogen.GroupChat(
    agents=[admin, assistant, writer, critic],
    messages=[],
    max_round=6,                       # [02번]의 MAX_REVISIONS와 동일하게 무한 루프를 방지한다
    speaker_selection_method="auto",   # GroupChatManager(LLM)가 매 턴 다음 발언자를 고른다
)
manager = autogen.GroupChatManager(groupchat=groupchat, llm_config=llm_config)

admin.initiate_chat(manager, message="온디바이스 AI 반도체에 대한 3문단 보고서를 만들어줘.")
# critic이 'APPROVED'라고 답할 때까지, 혹은 max_round에 도달할 때까지 대화가 이어진다

Admin (to chat_manager):

온디바이스 AI 반도체에 대한 3문단 보고서를 만들어줘.

--------------------------------------------------------------------------------

Next speaker: Researcher

Researcher (to chat_manager):

### 온디바이스 AI 반도체 개요

온디바이스 AI 반도체는 데이터 처리와 인공지능 알고리즘을 기기 내부에서 직접 수행할 수 있도록 설계된 반도체 칩을 의미합니다. 이러한 기술은 클라우드 기반 처리의 의존도를 줄이고, 실시간 데이터 분석과 응답 속도를 향상시키는 데 기여합니다. 특히, IoT 기기, 스마트폰, 자율주행차 등 다양한 분야에서 활용되며, 사용자 경험을 개선하고 보안성을 높이는 데 중요한 역할을 하고 있습니다.

### 시장 성장 동력

온디바이스 AI 반도체 시장은 인공지능 기술의 발전과 함께 급속히 성장하고 있습니다. 특히, 데이터 프라이버시와 보안에 대한 우려가 커짐에 따라, 사용자 데이터가 외부 서버로 전송되지 않고 기기 내에서 처리되는 솔루션에 대한 수요가 증가하고 있습니다. 또한, 5G와 같은 고속 통신 기술의 발전은 실시간 데이터 처리의 필요성을 더욱 부각시키고 있으며, 이는 온디바이스 AI 반도체의 채택을 가속화하고 있습니다.

### 주요 기업 및 기술 동향

현재 온디바이스 AI 반도체 시장에는 NVIDIA, Google, Intel, Qualcomm 등 주요 기술 기업들이 활발히 참여하고 있습니다. 이들은 고성능의 AI 연산을 지원하는 전용 칩을 개발하고 있으며, 에너지 효율성을 극대화하는 방향으로 기술 혁신을 추진하고 있습니다. 또한, 머신러닝 모델의 경량화와 최적화가 이루어지면서, 다양한 애플리케이션에서 온디바이스 AI의 활용 가능성이 더욱 확대되고 있습니다. 이러한 트렌드는 앞으로도 지속될 것으로 예상되며, 시장 경쟁이 치열해질 것입니다.

-------------------

ChatResult(chat_id=308117280621926521115465461222500202050, chat_history=[{'content': '온디바이스 AI 반도체에 대한 3문단 보고서를 만들어줘.', 'role': 'assistant', 'name': 'Admin'}, {'content': '### 온디바이스 AI 반도체 개요\n\n온디바이스 AI 반도체는 데이터 처리와 인공지능 알고리즘을 기기 내부에서 직접 수행할 수 있도록 설계된 반도체 칩을 의미합니다. 이러한 기술은 클라우드 기반 처리의 의존도를 줄이고, 실시간 데이터 분석과 응답 속도를 향상시키는 데 기여합니다. 특히, IoT 기기, 스마트폰, 자율주행차 등 다양한 분야에서 활용되며, 사용자 경험을 개선하고 보안성을 높이는 데 중요한 역할을 하고 있습니다.\n\n### 시장 성장 동력\n\n온디바이스 AI 반도체 시장은 인공지능 기술의 발전과 함께 급속히 성장하고 있습니다. 특히, 데이터 프라이버시와 보안에 대한 우려가 커짐에 따라, 사용자 데이터가 외부 서버로 전송되지 않고 기기 내에서 처리되는 솔루션에 대한 수요가 증가하고 있습니다. 또한, 5G와 같은 고속 통신 기술의 발전은 실시간 데이터 처리의 필요성을 더욱 부각시키고 있으며, 이는 온디바이스 AI 반도체의 채택을 가속화하고 있습니다.\n\n### 주요 기업 및 기술 동향\n\n현재 온디바이스 AI 반도체 시장에는 NVIDIA, Google, Intel, Qualcomm 등 주요 기술 기업들이 활발히 참여하고 있습니다. 이들은 고성능의 AI 연산을 지원하는 전용 칩을 개발하고 있으며, 에너지 효율성을 극대화하는 방향으로 기술 혁신을 추진하고 있습니다. 또한, 머신러닝 모델의 경량화와 최적화가 이루어지면서, 다양한 애플리케이션에서 온디바이스 AI의 활용 가능성이 더욱 확대되고 있습니다. 이러한 트렌드는 앞으로도 지속될 것으로 예상되며, 시장 경쟁이 치열해질 것입니다.', 'name': 'Researcher', 'role': 'us

### 정리

| 우리가 직접 만든 것 (01~03번) | CrewAI | AutoGen (AG2) |
|---|---|---|
| 고정 순서로 노드를 잇는 `StateGraph` | `Process.sequential` | `GroupChat(speaker_selection_method="round_robin")` |
| `manager_node`의 `Decision` 구조화 출력 | `Process.hierarchical` + `manager_llm` | `GroupChatManager(speaker_selection_method="auto")` |
| `Message(sender, receiver, type, content)` | Task의 `context=[...]`로 결과물 전달 | Agent 간 대화 메시지 자체 |
| `MAX_REVISIONS`로 루프 제한 | (Task 재시도 설정) | `GroupChat(max_round=...)` |

- **CrewAI**는 "역할과 작업을 선언하면 실행 순서는 프레임워크가 알아서"라는 철학이다. 파이프라인이 비교적 명확한 업무(조사→작성→검토)에 빠르게 적용하기 좋다.
- **AutoGen**은 "에이전트들을 대화방에 넣고 대화가 스스로 문제를 해결하게 한다"는 철학이다. 자유도가 높고 사람이 대화에 끼어들기 쉬워서(`UserProxyAgent`), 브레인스토밍이나 사람 승인이 필요한 워크플로에 강하다.
- **LangGraph**(01~03번에서 쓴 도구)는 이 둘보다 한 단계 낮은 수준이다. `Process`나 `GroupChatManager` 같은 기본 제공 전략이 없는 대신, 라우팅 조건과 State 구조를 원하는 대로 세밀하게 제어할 수 있다 — 이번 노트북들에서 `manager_node`, `route_by_message` 등을 직접 코드로 짠 이유다.
- 실무에서는 "표준적인 협업 흐름은 CrewAI/AutoGen으로 빠르게 프로토타이핑하고, 세밀한 제어가 필요한 구간만 LangGraph로 내려가 직접 구현"하는 조합도 흔하다.